# 🔬 TrustOCT-KD: Retinal OCT Disease Classification & Knowledge Distillation
## Calibration-Aware Knowledge Distillation with Explainability Preservation

**Repository**: [https://github.com/Gnanapravallika/TrustOCT-KD.git](https://github.com/Gnanapravallika/TrustOCT-KD.git)

---

## SECTION 1: Environment Setup
Prints GPU name, VRAM, PyTorch version, and installs required packages.

In [ ]:
!nvidia-smi

import torch
print(f"\n✅ PyTorch Version: {torch.__version__}")
print(f"✅ CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"✅ VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
!pip install -q kagglehub thop seaborn scikit-learn matplotlib tqdm opencv-python pandas
print("\n✅ All packages installed successfully!")

## SECTION 2: Clone Repository
Clones `Gnanapravallika/TrustOCT-KD` into Colab working directory.

In [ ]:
import os

if not os.path.exists('TrustOCT-KD'):
    !git clone https://github.com/Gnanapravallika/TrustOCT-KD.git

%cd TrustOCT-KD
print("\n✅ Repository cloned! Directory contents:")
!ls -la

## SECTION 3: Dataset Download
Configures Kaggle API & downloads Kermany dataset directly to Colab VM disk (~5GB, no Google Drive needed).

In [ ]:
# Upload your kaggle.json file
from google.colab import files
uploaded = files.upload()

import os
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
print("\n✅ Kaggle API key configured!")

In [ ]:
# Download dataset directly to Colab VM disk
!kaggle datasets download -d paultimothymooney/kermany2018 -p data/ --unzip
print("\n✅ Dataset downloaded to Colab disk!")
!du -sh data/

## SECTION 4: Dataset Exploration
Prints split class distributions table and displays an inline 2x4 visual image grid of sample OCT scans (CNV, DME, Drusen, Normal).

In [ ]:
import sys, os
sys.path.append('.')

from trustoct.dataset import print_class_distributions

# Find dataset root directory
DATA_DIR = 'data'
for root, dirs, f in os.walk(DATA_DIR):
    if 'train' in dirs and 'test' in dirs:
        DATA_DIR = root
        break

# Print Split Class Distribution breakdown table
print_class_distributions(DATA_DIR)

In [ ]:
# Display 2x4 visual image grid of sample OCT scans per class
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np

classes = ['CNV', 'DME', 'DRUSEN', 'NORMAL']
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for i, cls in enumerate(classes):
    cls_dir = os.path.join(DATA_DIR, 'train', cls)
    imgs = [f for f in os.listdir(cls_dir) if f.lower().endswith(('.jpeg','.jpg','.png'))]
    
    for j in range(2):
        img = Image.open(os.path.join(cls_dir, imgs[j]))
        axes[j, i].imshow(np.array(img), cmap='gray')
        axes[j, i].set_title(f'{cls}', fontsize=14, fontweight='bold')
        axes[j, i].axis('off')

plt.suptitle('Sample Retinal OCT Scans per Class (CNV, DME, DRUSEN, NORMAL)', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()
print("\n✅ Dataset exploration complete!")

## SECTION 5: Global Training Hyperparameters

In [ ]:
# Hyperparameters
epochs = 20
lr = 1e-4
batch_size = 32

print(f"⚙️ Configured Hyperparameters:")
print(f"   Epochs:        {epochs}")
print(f"   Learning Rate: {lr}")
print(f"   Batch Size:    {batch_size}")

## SECTION 6: Model Training Execution & Metrics Printout

Trains the 3 core models separately under identical hyperparameters, printing live epoch loss, accuracy, F1 score, and `✅ Best model updated!` status.

### 1. Train `msf_cbam_resnet50` (+ MultiScale + CBAM - Teacher Model)

In [ ]:
from trustoct.training.trainer import run_experiment
from trustoct.evaluation.metrics import evaluate_classification
from trustoct.dataset.oct_dataset import get_dataloaders
from trustoct.models import build_model
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
train_loader, val_loader, test_loader = get_dataloaders(DATA_DIR, batch_size=batch_size, num_workers=2)

# 1. Train Teacher Model
teacher_ckpt, teacher_hist = run_experiment('resnet50_msf_cbam', DATA_DIR, epochs=epochs, lr=lr, batch_size=batch_size)

# 2. Print Test Set Metrics
teacher_model = build_model('resnet50_msf_cbam', num_classes=4, pretrained=False)
ckpt = torch.load(teacher_ckpt, map_location=device)
teacher_model.load_state_dict(ckpt['model_state_dict'])
teacher_model = teacher_model.to(device)

teacher_metrics, t_labels, t_preds, t_probs, t_cm = evaluate_classification(teacher_model, test_loader, device)
print("\n" + "="*45)
print(" 📊 TEST METRICS: msf_cbam_resnet50 (Teacher)")
print("="*45)
for k, v in teacher_metrics.items():
    print(f"  {k:<30} {v*100:.2f}%" if v <= 1 else f"  {k:<30} {v:.4f}")

### 2. Train `student_mobilenetv3` (Student Baseline w/o KD)

In [ ]:
from trustoct.models import build_student

# 1. Train Model
student_no_kd_ckpt, student_no_kd_hist = run_experiment('student_mobilenetv3', DATA_DIR, epochs=epochs, lr=1e-3, batch_size=batch_size)

# 2. Print Test Set Metrics
student_no_kd_model = build_student('mobilenetv3', num_classes=4, pretrained=False)
ckpt = torch.load(student_no_kd_ckpt, map_location=device)
student_no_kd_model.load_state_dict(ckpt['model_state_dict'])
student_no_kd_model = student_no_kd_model.to(device)

sn_metrics, sn_labels, sn_preds, sn_probs, sn_cm = evaluate_classification(student_no_kd_model, test_loader, device)
print("\n" + "="*45)
print(" 📊 TEST METRICS: student_mobilenetv3 (No KD)")
print("="*45)
for k, v in sn_metrics.items():
    print(f"  {k:<30} {v*100:.2f}%" if v <= 1 else f"  {k:<30} {v:.4f}")

### 3. Train `student_kd_mobilenetv3` (+ Calibration-Aware KD - Proposed)

In [ ]:
from trustoct.training.distillation_trainer import DistillationTrainer

# Instantiate student
student_kd_model = build_student('mobilenetv3', num_classes=4, pretrained=True)

# Calibration-aware Distillation
distiller = DistillationTrainer(
    teacher_model=teacher_model,
    student_model=student_kd_model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    lr=1e-3,
    num_epochs=epochs,
    temperature=4.0,
    alpha=0.3, beta=0.5, gamma=0.2,
    experiment_name='student_KD_TrustOCT'
)

student_kd_ckpt, kd_hist = distiller.fit()

# Print Test Set Metrics
ckpt = torch.load(student_kd_ckpt, map_location=device)
student_kd_model.load_state_dict(ckpt['model_state_dict'])
student_kd_model = student_kd_model.to(device)

sk_metrics, sk_labels, sk_preds, sk_probs, sk_cm = evaluate_classification(student_kd_model, test_loader, device)
print("\n" + "="*45)
print(" 📊 TEST METRICS: student_kd_mobilenetv3 (Proposed)")
print("="*45)
for k, v in sk_metrics.items():
    print(f"  {k:<30} {v*100:.2f}%" if v <= 1 else f"  {k:<30} {v:.4f}")

## SECTION 7: Evaluate Teacher Model & Display Confusion Matrix

In [ ]:
from trustoct.evaluation.metrics import plot_confusion_matrix
from IPython.display import Image, display

plot_confusion_matrix(t_cm, classes, save_path='outputs/visualizations/teacher_confusion_matrix.png',
                      title='Teacher (ResNet50+MSF+CBAM) — Confusion Matrix')
display(Image(filename='outputs/visualizations/teacher_confusion_matrix.png', width=500))

## SECTION 8: Evaluate Student Models (No KD vs WITH KD)

In [ ]:
plot_confusion_matrix(sn_cm, classes, save_path='outputs/visualizations/student_no_kd_confusion_matrix.png',
                      title='Student (No KD) — Confusion Matrix')
display(Image(filename='outputs/visualizations/student_no_kd_confusion_matrix.png', width=500))

plot_confusion_matrix(sk_cm, classes, save_path='outputs/visualizations/student_kd_confusion_matrix.png',
                      title='Student WITH KD (Proposed) — Confusion Matrix')
display(Image(filename='outputs/visualizations/student_kd_confusion_matrix.png', width=500))

## SECTION 9: MASTER SUMMARY COMPARISON TABLE (All 5 Categories Combined)

Merges **Discrimination** (Acc, F1, MCC, Kappa, ROC-AUC), **Calibration** (ECE %, Brier Score), **Explainability Faithfulness** (Deletion & Insertion AOPC), **Efficiency** (Params, Size, Latency, FPS), and **Robustness** into one single Master Paper Table!

In [ ]:
from trustoct.evaluation.comparison import run_full_comparison
from IPython.display import display

# Run and display Master Summary Comparison Table
df_master = run_full_comparison(
    teacher_model=teacher_model,
    student_model=student_kd_model,
    student_no_kd_model=student_no_kd_model,
    test_loader=test_loader,
    device=device
)

# Render interactive HTML table in Colab
display(df_master)

## SECTION 10: Calibration Analysis
Prints ECE % & Brier Score + plots side-by-side Reliability Diagrams.

In [ ]:
from trustoct.evaluation.calibration import compute_calibration_metrics, plot_reliability_diagram

t_calib = compute_calibration_metrics(t_labels, t_probs)
sn_calib = compute_calibration_metrics(sn_labels, sn_probs)
sk_calib = compute_calibration_metrics(sk_labels, sk_probs)

print("\n" + "="*60)
print(" 📊 CALIBRATION ANALYSIS")
print("="*60)
print(f"{'Model':<35} {'ECE (%)':>10} {'Brier':>10}")
print("-" * 60)
print(f"{'Teacher (ResNet50+MSF+CBAM)':<35} {t_calib['ECE']*100:>9.2f}% {t_calib['Brier_Score']:>10.4f}")
print(f"{'Student w/o KD (MobileNetV3)':<35} {sn_calib['ECE']*100:>9.2f}% {sn_calib['Brier_Score']:>10.4f}")
print(f"{'Student w/ KD (Ours)':<35} {sk_calib['ECE']*100:>9.2f}% {sk_calib['Brier_Score']:>10.4f}")

In [ ]:
# Plot side-by-side Reliability Diagrams
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, calib) in zip(axes, [
    ('Teacher', t_calib),
    ('Student w/o KD', sn_calib),
    ('Student w/ KD (Ours)', sk_calib)
]):
    num_bins = len(calib['bin_accs'])
    bin_centers = [0.05 + i * 0.1 for i in range(num_bins)]
    ax.bar(bin_centers, calib['bin_accs'], width=0.08, alpha=0.7, color='#2b5c8f', edgecolor='black')
    ax.plot([0, 1], [0, 1], 'k--', linewidth=2)
    ax.set_title(f'{name}\nECE: {calib["ECE"]*100:.2f}% | Brier: {calib["Brier_Score"]:.4f}', fontweight='bold', fontsize=12)
    ax.set_xlabel('Confidence'); ax.set_ylabel('Accuracy')
    ax.set_xlim([0, 1]); ax.set_ylim([0, 1])
    ax.grid(True, linestyle=':')

plt.suptitle('Reliability Diagrams: Teacher vs Student Calibration', fontsize=14, fontweight='bold', y=1.03)
plt.tight_layout()
plt.savefig('outputs/visualizations/reliability_diagrams_comparison.png', dpi=300, bbox_inches='tight')
plt.show()
print("\n✅ Reliability diagrams figure saved!")

## SECTION 11: LayerCAM Explainability
Displays a 4x3 visual grid comparing original OCT vs Teacher LayerCAM vs Student LayerCAM.

In [ ]:
from trustoct.evaluation.explainability import LayerCAM, overlay_cam_on_image
import numpy as np

norm_mean = np.array([0.485, 0.456, 0.406]).reshape(1, 1, 3)
norm_std = np.array([0.229, 0.224, 0.225]).reshape(1, 1, 3)

teacher_cam = LayerCAM(teacher_model, target_layer=teacher_model.cbam_fused)
student_cam = LayerCAM(student_kd_model, target_layer=student_kd_model.features[-1])

samples = {}
for images, labels in test_loader:
    for img, lbl in zip(images, labels):
        cls_idx = lbl.item()
        if cls_idx not in samples:
            samples[cls_idx] = img
        if len(samples) >= 4:
            break
    if len(samples) >= 4:
        break

fig, axes = plt.subplots(4, 3, figsize=(14, 18))

for row, (cls_idx, img_tensor) in enumerate(sorted(samples.items())):
    cls_name = classes[cls_idx]
    img_input = img_tensor.unsqueeze(0).to(device)
    img_np = img_tensor.cpu().numpy().transpose(1, 2, 0)
    img_np = np.clip(img_np * norm_std + norm_mean, 0, 1)

    t_map, t_pred, t_prob = teacher_cam.generate(img_input, target_class=cls_idx)
    s_map, s_pred, s_prob = student_cam.generate(img_input, target_class=cls_idx)

    axes[row, 0].imshow(img_np)
    axes[row, 0].set_title(f'Original ({cls_name})', fontweight='bold', fontsize=12)
    axes[row, 0].axis('off')

    axes[row, 1].imshow(overlay_cam_on_image(img_np, t_map))
    axes[row, 1].set_title(f'Teacher CAM ({t_prob*100:.1f}%)', fontweight='bold', fontsize=12)
    axes[row, 1].axis('off')

    axes[row, 2].imshow(overlay_cam_on_image(img_np, s_map))
    axes[row, 2].set_title(f'Student CAM ({s_prob*100:.1f}%)', fontweight='bold', fontsize=12)
    axes[row, 2].axis('off')

plt.suptitle('LayerCAM Visual Explainability: Teacher vs Student', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('outputs/visualizations/layercam_comparison.png', dpi=300, bbox_inches='tight')
plt.show()
print("\n✅ LayerCAM explainability grid saved!")

## SECTION 12: AOPC Faithfulness
Evaluates quantitative Deletion/Insertion AOPC scores + plots confidence decay curves.

In [ ]:
from trustoct.evaluation.explainability import compute_aopc_faithfulness

print("\n" + "="*60)
print(" 📊 AOPC FAITHFULNESS EVALUATION")
print("="*60)
print(f"{'Class':<10} {'Teacher Del-AOPC':>18} {'Student Del-AOPC':>18} {'Teacher Ins-AOPC':>18} {'Student Ins-AOPC':>18}")
print("-" * 85)

aopc_rows = []
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for idx, (cls_idx, img_tensor) in enumerate(sorted(samples.items())):
    cls_name = classes[cls_idx]
    img_input = img_tensor.unsqueeze(0).to(device)

    t_map, _, _ = teacher_cam.generate(img_input, target_class=cls_idx)
    s_map, _, _ = student_cam.generate(img_input, target_class=cls_idx)

    t_aopc = compute_aopc_faithfulness(teacher_model, img_input, t_map, cls_idx, device=device)
    s_aopc = compute_aopc_faithfulness(student_kd_model, img_input, s_map, cls_idx, device=device)

    print(f"{cls_name:<10} {t_aopc['deletion_aopc']:>18.4f} {s_aopc['deletion_aopc']:>18.4f} {t_aopc['insertion_aopc']:>18.4f} {s_aopc['insertion_aopc']:>18.4f}")

    aopc_rows.append({'Class': cls_name, 'Teacher Del-AOPC': t_aopc['deletion_aopc'],
                      'Student Del-AOPC': s_aopc['deletion_aopc'],
                      'Teacher Ins-AOPC': t_aopc['insertion_aopc'],
                      'Student Ins-AOPC': s_aopc['insertion_aopc']})

    ax = axes[idx // 2, idx % 2]
    ax.plot(t_aopc['percentages'], t_aopc['deletion_scores'], 'r-o', label=f'Teacher (AOPC={t_aopc["deletion_aopc"]:.3f})', markersize=4)
    ax.plot(s_aopc['percentages'], s_aopc['deletion_scores'], 'b-s', label=f'Student (AOPC={s_aopc["deletion_aopc"]:.3f})', markersize=4)
    ax.set_title(f'{cls_name} — Deletion Curve', fontweight='bold')
    ax.set_xlabel('% Pixels Masked'); ax.set_ylabel('Confidence')
    ax.legend(fontsize=9); ax.grid(True, linestyle=':')

plt.suptitle('AOPC Faithfulness: Teacher vs Student', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('outputs/visualizations/aopc_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

aopc_df = pd.DataFrame(aopc_rows)
aopc_df.to_csv('outputs/results/aopc_comparison.csv', index=False)
print("\n✅ AOPC faithfulness comparison saved!")

## SECTION 13: Clinical Robustness
Tests and prints performance decay under noise, brightness, and contrast corruptions.

In [ ]:
from trustoct.evaluation.robustness import evaluate_robustness

t_robust = evaluate_robustness(teacher_model, test_loader, device=device)
sn_robust = evaluate_robustness(student_no_kd_model, test_loader, device=device)
sk_robust = evaluate_robustness(student_kd_model, test_loader, device=device)

print("\n" + "="*80)
print(" 📊 CLINICAL ROBUSTNESS EVALUATION")
print("="*80)
print(f"{'Perturbation':<30} {'Teacher':>12} {'No KD':>12} {'KD (Ours)':>12}")
print("-" * 70)

for perturbation in t_robust.keys():
    t_acc = t_robust[perturbation]['Accuracy'] * 100
    sn_acc = sn_robust[perturbation]['Accuracy'] * 100
    sk_acc = sk_robust[perturbation]['Accuracy'] * 100
    print(f"{perturbation:<30} {t_acc:>11.2f}% {sn_acc:>11.2f}% {sk_acc:>11.2f}%")

print("\n✅ Clinical robustness evaluation complete!")

## SECTION 14: Model Complexity
Profiles parameter count (33M vs 1.2M), disk footprint, inference latency (ms), and FPS.

In [ ]:
from trustoct.evaluation.benchmark import profile_model_complexity

t_complexity = profile_model_complexity(teacher_model, device=device)
sk_complexity = profile_model_complexity(student_kd_model, device=device)

print("\n" + "="*60)
print(" 📊 MODEL COMPLEXITY & LATENCY COMPARISON")
print("="*60)
print(f"{'Metric':<25} {'Teacher':>18} {'Student (Ours)':>18}")
print("-" * 60)
for key in t_complexity:
    print(f"{key:<25} {t_complexity[key]:>18} {sk_complexity[key]:>18}")

print("\n✅ Model complexity profiling complete!")

## SECTION 15: Export Results
Automatically zips all generated CSV tables and PNG figures for a 1-click download to your laptop.

In [ ]:
# Option A: Download ZIP to laptop
!zip -r TrustOCT_Results.zip outputs/results/ outputs/visualizations/

from google.colab import files
files.download('TrustOCT_Results.zip')
print("\n✅ All results ZIP downloaded to your laptop!")

In [ ]:
# Option B: Save to Google Drive (optional)
from google.colab import drive
drive.mount('/content/drive')

import shutil, glob
save_dir = '/content/drive/MyDrive/TrustOCT_Results'
os.makedirs(save_dir, exist_ok=True)

for folder in ['results', 'visualizations']:
    src = f'outputs/{folder}'
    if os.path.exists(src):
        shutil.copytree(src, f'{save_dir}/{folder}', dirs_exist_ok=True)

for ckpt in glob.glob('outputs/checkpoints/*_best.pth'):
    shutil.copy(ckpt, save_dir)

print(f"\n✅ All results and checkpoints saved to Google Drive: {save_dir}")